<a href="https://colab.research.google.com/github/luizaureliobn/clust-learn/blob/master/notebooks/clustering_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# `clust-learn` - Module 3: Clustering

This guide shows how to use the `clustering` module of the `clust-learn` package to compute and explain clusters.

## 0. Setup

In [1]:
import numpy as np
import pandas as pd

from clearn.clustering import Clustering
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

ModuleNotFoundError: No module named 'clearn'

## 1. Data loading

DataFrame with extracted dimensions.

In [ ]:
df = pd.read_csv('data/pisa_spain_sample_v2_preprocessed_dim_red_output.csv')
print(df.shape)
df.head()

DataFrame with original variables. This will later be used for results interpretation.

In [ ]:
df_original = pd.read_csv('data/pisa_spain_sample_v2_preprocess_ouput.csv')
print(df_original.shape)
df_original.head()

We separate numerical and categorical variables from the original set of variables.

In [ ]:
num_vars = ['AGE', 'PAREDINT', 'BMMJ1',
       'BFMJ2', 'HISEI', 'DURECEC', 'BSMJ', 'MMINS',
       'LMINS', 'SMINS', 'TMINS', 'FCFMLRTY', 'SCCHANGE', 'CHANGE', 'STUBMI',
       'ESCS', 'UNDREM', 'METASUM', 'METASPAM', 'ICTHOME', 'ICTSCH', 'HOMEPOS',
       'CULTPOSS', 'HEDRES', 'WEALTH', 'ICTRES', 'DISCLIMA', 'TEACHSUP',
       'DIRINS', 'PERFEED', 'EMOSUPS', 'STIMREAD', 'ADAPTIVITY', 'TEACHINT',
       'JOYREAD', 'SCREADCOMP', 'SCREADDIFF', 'PERCOMP', 'PERCOOP', 'ATTLNACT',
       'COMPETE', 'WORKMAST', 'GFOFAIL', 'EUDMO', 'SWBP', 'RESILIENCE',
       'MASTGOAL', 'GCSELFEFF', 'GCAWARE', 'ATTIMM', 'INTCULT', 'PERSPECT',
       'COGFLEX', 'RESPECT', 'AWACOM', 'GLOBMIND', 'DISCRIM', 'BELONG',
       'BEINGBULLIED', 'ENTUSE', 'HOMESCH', 'USESCH', 'INTICT', 'COMPICT',
       'AUTICT', 'SOIAICT', 'ICTCLASS', 'ICTOUTSIDE', 'INFOCAR', 'INFOJOB1',
       'INFOJOB2', 'FLCONFIN', 'FLCONICT', 'FLSCHOOL', 'FLFAMILY', 'BODYIMA',
       'SOCONPA']
cat_vars = ['ST004D01T', 'IMMIG', 'REPEAT']

We make a selection of the original variables for some visualizations.

In [ ]:
num_vars_sel = ['HISEI', 'MMINS', 'LMINS', 'FCFMLRTY', 'SCCHANGE', 'CHANGE',
       'ESCS', 'METASUM', 'ICTHOME', 'HOMEPOS', 'ICTRES', 'TEACHSUP',
       'EMOSUPS', 'STIMREAD', 'ADAPTIVITY', 'JOYREAD', 'SCREADCOMP', 'SCREADDIFF',
       'WORKMAST', 'GFOFAIL', 'RESILIENCE','MASTGOAL', 'PERSPECT',
       'RESPECT', 'GLOBMIND', 'DISCRIM','COMPICT', 'INFOCAR', 'FLSCHOOL', 'BODYIMA']

## 1. Clustering computation

We compare k-means, agglomerative clustering with Ward's linkage, and Gaussian Mixture models. We don't normalize the data (`normalize=False`) because we're using the components extracted from the dimensionality reduction process.

In [ ]:
km = KMeans(random_state=42)
ward = AgglomerativeClustering()
gmm = GaussianMixture()

In [ ]:
cl = Clustering(df, algorithms= [km, ward, gmm], normalize=False)

In [ ]:
cl.df.describe()

The code below computes the optimal number of clusters between 2 and 21 and adds the prefict `'STU'` (from the word student) to the cluster labels.

As performance metric teh default one is used (WSS or inertia - `metric='inertia'`).

In [ ]:
cl.compute_clusters(max_clusters = 21, prefix='STU')

In [ ]:
cl.scores_

In [ ]:
cl.labels_

In [ ]:
cl.metric_

In [ ]:
cl.optimal_config_

## 2. Cluster performance

The code below generates a bar plot with the number of observations per cluster.

In [ ]:
cl.plot_clustercount() #output_path='PATH.jpg')

The code below generates a plot with the performance comparison between kmeans and aggomerative clustering.

In [ ]:
cl.plot_score_comparison()

The code below plots the normalized WSS and difference curve used by the elbow method.

In [ ]:
cl.plot_optimal_components_normalized() #output_path='PATH.jpg')

## 3. Analysis of clusters

### 3.1. Descriptive statistics by cluster

#### 3.1.1. Internal variables

In [ ]:
# Internal variables
cl.describe_clusters(variables=['dim_01', 'dim_12'], statistics='mean')

In [ ]:
cl.describe_clusters()

#### 3.1.2. External variables

In [ ]:
# External variables - We use (some of) the original variables
cl.describe_clusters(df_original[num_vars], variables=num_vars, statistics='mean')

In [ ]:
# Use the original variables - Some of them
cl.describe_clusters_cat(df_original['REPEAT'], cat_name='REPEAT', normalize=True)

In [ ]:
cl.describe_clusters_cat(df_original['ST004D01T'], cat_name='ST004D01T', normalize=True)

### 3.2. Cluster means vs global means comparison

#### 3.2.1. Internal variables

In [ ]:
cl.compare_cluster_means_to_global_means()

In [ ]:
cl.plot_cluster_means_to_global_means_comparison(xlabel='Principal Components', ylabel='Clusters',
                                                 levels=[-1, -0.67, -0.3, -0.15, 0.15, 0.3, 0.67, 1])

#### 3.2.2. External variables

Note that for external variables we normalize variables to the 0-1 scale in order to make fair comparisons.

In [ ]:
mms = MinMaxScaler()

In [ ]:
cl.compare_cluster_means_to_global_means(pd.DataFrame(mms.fit_transform(df_original[num_vars_sel]), columns=num_vars_sel))

In [ ]:
cl.plot_cluster_means_to_global_means_comparison(df_original=pd.DataFrame(mms.fit_transform(df_original[num_vars_sel]), columns=num_vars_sel),
                                                 xlabel='Principal Components', ylabel='Clusters', use_weights=True)

### 3.3. Significance tests

In [ ]:
# ANOVA tests for internal variables
cl.anova_tests(cluster_filter=[4, 5], vars_test=['dim_01', 'dim_05', 'dim_12'])

In [ ]:
cl.anova_tests(cluster_filter=[4, 5], vars_test=['dim_02', 'dim_03'])

In [ ]:
# ANOVA tests for External variables
cl.anova_tests(cluster_filter=[4, 5], df_test=df_original[['HOMEPOS', 'ESCS']])

In [ ]:
cl.chi2_test(df_original['ST004D01T'])

In [ ]:
cl.chi2_test(df_original['REPEAT'])

In [ ]:
cl.chi2_test(df_original['IMMIG'])

### 3.4. Distribution comparisons (visualizations) for numerical variables

#### 3.4.1. Internal variables

In [ ]:
# Internal variables
cl.plot_distribution_comparison_by_cluster()

#### 3.4.2. External variables

In [ ]:
# External variables
cl.plot_distribution_comparison_by_cluster(df_ext=df_original[['HOMEPOS', 'ESCS']])

In [ ]:
# External variables
cl.plot_distribution_comparison_by_cluster(df_ext=df_original[['ESCS', 'MASTGOAL']])

In [ ]:
cl.plot_distribution_comparison_by_cluster(df_ext=df_original[['ESCS', 'TEACHSUP']]) #, output_path='PATH.jpg')

### 3.5. 2-Dimensional plots for numerical variables

In [ ]:
# Internal variables
cl.plot_clusters_2D('dim_01', 'dim_02', style_kwargs=dict(kdeplot=True, alpha=0.2)) #, output_path='PATH.jpg')

In [ ]:
#External Variables
cl.plot_clusters_2D(df_original['HOMEPOS'], df_original['ESCS'])

### 3.6. Distribution comparisons (visualization) for categorical variables

In [ ]:
cl.plot_cat_distribution_by_cluster(df_original['REPEAT'], cat_label='REPEAT', cluster_label='Student clusters')

In [ ]:
cl.plot_cat_distribution_by_cluster(df_original['ST004D01T'], cat_label='ST004D01T', cluster_label='Student clusters')

In [ ]:
cl.plot_cat_distribution_by_cluster(df_original['IMMIG'], cat_label='IMMIG', cluster_label='Student clusters')

## Export results

Export results:
- One data set with the clusters associated to the extracted components;
- Another one with the clusters associated to the original data.

In [ ]:
# cl.df.to_csv('../../data/pisa_spain_sample_v2_preprocessed_dim_red_clustered_output.csv', index=False)

In [ ]:
# df_original['cluster'] = cl.df['cluster'].values
# df_original['cluster_cat'] = cl.df['cluster_cat'].values
# df_original.to_csv('../../data/pisa_spain_sample_v2_preprocessed_clustered_output.csv', index=False)

## Reproducibility

In [ ]:
df_ref = pd.read_csv('data/pisa_spain_sample_v2_preprocessed_dim_red_clustered_output.csv')

print('Diff', (df_ref['cluster'] != cl.df['cluster']).sum())